# 4. Attention analysis
Deletion tests (is attention faithful?) and attention statistics (pitch, duration, measure position, rests).

Run `1_train.ipynb` and `2_ablations_and_controls.ipynb` first: this notebook uses the trained models `full` and `no_meas`.

In [ ]:
import torch

from constants import *
from utils import get_device
from faithfulness import run_deletion_tests
from attention_inspector import AttentionInspector

In [ ]:
USE_GPU = True
device = get_device(USE_GPU)
print(device)

# QUICK_TEST = True uses the models of the dry run in results_quick_test/
# and one deletion fraction only.
QUICK_TEST = False

if QUICK_TEST:
    results_dir = QUICK_TEST_RESULTS_DIR
    deletion_fractions = [0.10]
    num_random_draws = 1
else:
    results_dir = RESULTS_DIR
    deletion_fractions = DELETION_FRACTIONS
    num_random_draws = NUM_RANDOM_DRAWS

## Deletion tests
A fraction of the notes of every test piece is deleted and the piece is classified again. Four deletion orders: most attended notes first, least attended first, highest saliency first, random.

In [ ]:
curves, deletion_summary = run_deletion_tests(
    tag="full",
    device=device,
    seeds=SEEDS,
    fractions=deletion_fractions,
    num_random_draws=num_random_draws,
    results_dir=results_dir
)

In [ ]:
deletion_summary

## Attention statistics of the proposed model
First the note table is collected: one row per note of every test piece, with its attention weight.

In [ ]:
ai = AttentionInspector(device, results_dir=results_dir)

In [ ]:
note_table = ai.collect_note_table("full", seeds=SEEDS)
note_table.head()

Pitch enrichment, duration, measure position and rests. The report is written to `attention/full_report.json`.

In [ ]:
report, enrichment = ai.build_report("full", note_table)
ai.print_report(report, enrichment)

Share of notes, share of attention and enrichment of the most frequent pitches of one makam.

In [ ]:
makam = "hicaz"
enrichment[enrichment["makam"] == makam].head(10)

## The same statistics for the model without the measure-position input
If attention still differs between the start, the middle and the end of the measure, the difference does not come from the measure-position input.

In [ ]:
note_table_no_meas = ai.collect_note_table("no_meas", seeds=SEEDS)
report_no_meas, enrichment_no_meas = ai.build_report("no_meas", note_table_no_meas)
ai.print_report(report_no_meas, enrichment_no_meas)